In [2]:
import sqlite3

# Conecta ao banco de dados
conn = sqlite3.connect("sistema_estoque.db")
cursor = conn.cursor()


def executar_consulta(titulo, query):
    print("-" * 80)
    print(f"📊 {titulo}")
    print("-" * 80)
    cursor.execute(query)

    # Pega os nomes das colunas
    colunas = [col[0] for col in cursor.description]
    print(" | ".join(colunas))
    print("-" * 80)

    # Mostra os resultados (limitado a 5 para não lotar a tela)
    linhas = cursor.fetchall()
    for linha in linhas[:5]:  # <--- Corrigido aqui de 'lines' para 'linhas'
        print(" | ".join(str(item) for item in linha))

    if len(linhas) > 5:
        print(f"... e mais {len(linhas) - 5} linhas.")
    print("\n")

# --- CONSULTA 1: Ranking dos Clientes que Mais Gastaram ---
# (Agrupamento, Soma e Ordenação Decrescente)
q1 = """
SELECT 
    C.nome AS "Cliente",
    COUNT(DISTINCT P.id_pedido) AS "Qtd Pedidos",
    SUM(I.quantidade * I.preco_unitario) AS "Total Gasto (R$)"
FROM CLIENTE C
INNER JOIN PEDIDO P ON C.id_cliente = P.id_cliente
INNER JOIN ITEM_PEDIDO I ON P.id_pedido = I.id_pedido
GROUP BY C.id_cliente
ORDER BY "Total Gasto (R$)" DESC;
"""
executar_consulta("1. RANKING DE CLIENTES POR VALOR TOTAL COMPRADO", q1)


# --- CONSULTA 2: Faturamento Total e Quantidade de Vendas por Categoria ---
# (Análise de Performance de Produtos)
q2 = """
SELECT 
    PR.categoria AS "Categoria",
    COUNT(DISTINCT I.id_pedido) AS "Total de Vendas",
    SUM(I.quantidade) AS "Itens Vendidos",
    SUM(I.quantidade * I.preco_unitario) AS "Faturamento Bruto (R$)"
FROM PRODUTO PR
INNER JOIN ITEM_PEDIDO I ON PR.id_produto = I.id_produto
GROUP BY PR.categoria
ORDER BY "Faturamento Bruto (R$)" DESC;
"""
executar_consulta("2. DESEMPENHO COMERCIAL POR CATEGORIA DE PRODUTO", q2)


# --- CONSULTA 3: Auditoria de Usuários (Quem mais movimentou o estoque) ---
# (Uso de UNION para juntar históricos de tabelas diferentes)
q3 = """
SELECT 
    U.nome AS "Funcionário",
    U.tipo AS "Cargo",
    'Entrada' AS "Tipo Movimentação",
    SUM(E.quantidade) AS "Total Peças"
FROM USUARIO U
INNER JOIN ENTRADA E ON U.id_usuario = E.id_usuario
GROUP BY U.id_usuario

UNION ALL

SELECT 
    U.nome AS "Funcionário",
    U.tipo AS "Cargo",
    'Saída' AS "Tipo Movimentação",
    SUM(S.quantidade) AS "Total Peças"
FROM USUARIO U
INNER JOIN SAIDA S ON U.id_usuario = S.id_usuario
GROUP BY U.id_usuario
ORDER BY "Total Peças" DESC;
"""
executar_consulta(
    "3. AUDITORIA DE MOVIMENTAÇÃO DE ESTOQUE POR USUÁRIO (TOP 5)", q3
)


# --- CONSULTA 4: Lista de Produtos que Nunca Tiveram uma Saída Avulsa ---
# (Uso de Subconsulta com NOT IN para encontrar exceções/gargalos)
q4 = """
SELECT 
    id_produto AS "ID",
    nome AS "Produto Sem Saídas Avulsas",
    categoria AS "Categoria",
    preco AS "Preço (R$)"
FROM PRODUTO
WHERE id_produto NOT IN (SELECT DISTINCT id_produto FROM SAIDA)
ORDER BY nome ASC;
"""
executar_consulta("4. PRODUTOS SEM HISTÓRICO DE SAÍDAS AVULSAS (AVARIAS/PERDAS)", q4)


# --- CONSULTA 5: Resumo Mensal de Pedidos (Sazonalidade) ---
# (Tratamento de Strings de Data e Agrupamento por Ano/Mês)
q5 = """
SELECT 
    strftime('%Y-%m', data) AS "Ano-Mês",
    COUNT(id_pedido) AS "Qtd Pedidos",
    status AS "Status Atual"
FROM PEDIDO
GROUP BY "Ano-Mês", status
ORDER BY "Ano-Mês" DESC, "Qtd Pedidos" DESC;
"""
executar_consulta("5. EVOLUÇÃO MENSAL DE PEDIDOS E SEUS STATUS", q5)

# Fecha a conexão
conn.close()

--------------------------------------------------------------------------------
📊 1. RANKING DE CLIENTES POR VALOR TOTAL COMPRADO
--------------------------------------------------------------------------------
Cliente | Qtd Pedidos | Total Gasto (R$)
--------------------------------------------------------------------------------
Joaquim Fogaça | 5 | 11202.0
Yan Vasconcelos | 4 | 10301.28
Bella da Costa | 5 | 9471.0
Theo Ribeiro | 4 | 9315.05
Levi Rocha | 2 | 8480.41
... e mais 406 linhas.


--------------------------------------------------------------------------------
📊 2. DESEMPENHO COMERCIAL POR CATEGORIA DE PRODUTO
--------------------------------------------------------------------------------
Categoria | Total de Vendas | Itens Vendidos | Faturamento Bruto (R$)
--------------------------------------------------------------------------------
Beleza | 341 | 1188 | 323817.3
Eletrônicos | 323 | 1118 | 253204.92
Alimentos | 281 | 958 | 241864.46
Vestuário | 234 | 742 | 200105.12
H